In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [37]:
def rfeFeature(indep_X,dep_Y,n):
        rfelist=[]
        
        log_model = LogisticRegression(solver='lbfgs')
        RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        #NB = GaussianNB()
        DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
        svc_model = SVC(kernel = 'linear', random_state = 0)
        #KN=KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        rfemodellist=[log_model,svc_model,RF,DT]
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(i,n_features_to_select=n)
            log_fit = log_rfe.fit(indep_X, dep_Y)
            log_rfe_feature=log_fit.transform(indep_X)
            rfelist.append(log_rfe_feature)
        return rfelist

In [38]:
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        
        return X_train, X_test, y_train, y_test
    

In [39]:
def cm_prediction(classifier,X_test):
     y_pred = classifier.predict(X_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        #from sklearn.metrics import confusion_matrix
        #cm = confusion_matrix(y_test, y_pred)
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,X_test,y_test,cm


In [40]:
def logistic(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      
    
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
   
def Navie(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
def knn(X_train,y_train,X_test):
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      


def random(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

In [41]:
def rfe_classification(accLog, accSVM, accRF, accDT):

    rfedataframe = pd.DataFrame(
        index=['Logistic', 'SVC', 'Random', 'DecisionTree'],
        columns=['Logistic', 'SVC', 'Random', 'DecisionTree'],
        dtype=float
    )

    for number, index in enumerate(rfedataframe.index):

        rfedataframe.loc[index, 'Logistic'] = accLog[number]
        rfedataframe.loc[index, 'SVC'] = accSVM[number]
        rfedataframe.loc[index, 'Random'] = accRF[number]
        rfedataframe.loc[index, 'DecisionTree'] = accDT[number]

    return rfedataframe

In [42]:
dataset1=pd.read_csv("Elevator_predictive_maintenance_cleaned.csv",index_col=None)

df2=dataset1

df2 = pd.get_dummies(df2, drop_first=True,dtype=int)

indep_X=df2.drop('Maintenance_Required',axis=1)
dep_Y=df2['Maintenance_Required']



In [43]:
dep_Y=dep_Y.astype(int)

In [44]:
x=indep_X
print((x<0).sum())

Motor_Temperature_C       0
Motor_Vibration_mm_s      1
Motor_Current_A           0
Motor_Speed_RPM           0
Brake_Temperature_C       0
Door_Opening_Time_s       0
Door_Closing_Time_s       0
Load_Percentage           0
Number_of_Trips           1
Operating_Hours           1
Error_Count               1
Days_Since_Maintenance    1
Previous_Faults           1
dtype: int64


In [45]:
x=x.clip(lower=0) 

In [46]:
indep_X=x

In [47]:
import warnings
warnings.filterwarnings("ignore")

In [57]:
rfelist=rfeFeature(indep_X,dep_Y,7)       

accLog=[]
accSVM=[]
accsvmnl=[]
accknn=[]
accnav=[]
accDT=[]
accRF=[]

LogisticRegression()
SVC(kernel='linear', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)


In [58]:
for i in rfelist:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    
        
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
    accLog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
    accSVM.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
    accDT.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
    accRF.append(Accuracy)
    
 

 

In [59]:
result = rfe_classification(
    accLog,
    accSVM,
    accRF,
    accDT
)


In [52]:
result#6

,Logistic,SVC,Random,DecisionTree
Logistic,0.819672,0.844262,0.795082,0.754098
SVC,0.844262,0.852459,0.819672,0.819672
Random,0.844262,0.844262,0.827869,0.745902
DecisionTree,0.836066,0.844262,0.803279,0.737705


In [56]:
result #5

,Logistic,SVC,Random,DecisionTree
Logistic,0.827869,0.819672,0.795082,0.737705
SVC,0.852459,0.852459,0.811475,0.803279
Random,0.844262,0.852459,0.852459,0.737705
DecisionTree,0.836066,0.844262,0.844262,0.803279


In [60]:
result #7

,Logistic,SVC,Random,DecisionTree
Logistic,0.811475,0.844262,0.803279,0.704918
SVC,0.836066,0.844262,0.795082,0.762295
Random,0.827869,0.827869,0.770492,0.729508
DecisionTree,0.836066,0.844262,0.811475,0.770492
